In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================
# 1. IMPORT LIBRARIES
# ============================================

import os
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import confusion_matrix

In [ ]:
# ============================================
# 2. PARAMETERS
# ============================================

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30

train_path = "/content/drive/MyDrive/pneumonia/chest_xray/train"
val_path = "/content/drive/MyDrive/pneumonia/chest_xray/val"
test_path = "/content/drive/MyDrive/pneumonia/chest_xray/test"


In [ ]:
# ============================================
# 3. DATA PREPROCESSING (OPTIMIZED)
# ============================================

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    test_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

class_names = list(train_data.class_indices.keys())
print("Classes:", class_names)

Found 5219 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Classes: ['NORMAL', 'PNEUMONIA']


In [ ]:
# ============================================
# 4. LOAD RESNET50 BASE MODEL
# ============================================

base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze most layers
for layer in base_model.layers[:-140]:
    layer.trainable = False

# Unfreeze last layers
for layer in base_model.layers[-140:]:
    layer.trainable = True

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# ============================================
# 5. CUSTOM CLASSIFIER (OPTIMIZED)
# ============================================

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = BatchNormalization()(x)

x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
# ============================================
# 6. COMPILE MODEL
# ============================================

model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,155,009 (92.14 MB)

 Trainable params: 23,887,361 (91.12 MB)

 Non-trainable params: 267,648 (1.02 MB)

In [ ]:
# ============================================
# 7. CALLBACKS
# ============================================

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_model.h5",
    monitor='val_accuracy',
    save_best_only=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2,
    min_lr=1e-6
)


In [ ]:
# ============================================
# 8. TRAIN MODEL
# ============================================

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint, reduce_lr]
)


Epoch 1/30
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8388 - loss: 0.4000

164/164 ━━━━━━━━━━━━━━━━━━━━ 187s 1s/step - accuracy: 0.8396 - loss: 0.3929 - val_accuracy: 0.8125 - val_loss: 0.3662 - learning_rate: 1.0000e-05
Epoch 2/30
164/164 ━━━━━━━━━━━━━━━━━━━━ 0s 819ms/step - accuracy: 0.8742 - loss: 0.3447

164/164 ━━━━━━━━━━━━━━━━━━━━ 136s 828ms/step - accuracy: 0.8856 - loss: 0.3064 - val_accuracy: 0.9375 - val_loss: 0.2406 - learning_rate: 1.0000e-05
Epoch 3/30
164/164 ━━━━━━━━━━━━━━━━━━━━ 135s 822ms/step - accuracy: 0.9161 - loss: 0.2472 - val_accuracy: 0.9375 - val_loss: 0.1903 - learning_rate: 1.0000e-05
Epoch 4/30
164/164 ━━━━━━━━━━━━━━━━━━━━ 134s 816ms/step - accuracy: 0.9266 - loss: 0.2255 - val_accuracy: 0.9375 - val_loss: 0.1904 - learning_rate: 1.0000e-05
Epoch 5/30
164/164 ━━━━━━━━━━━━━━━━━━━━ 135s 820ms/step - accuracy: 0.9421 - loss: 0.1883 - val_accuracy: 0.9375 - val_loss: 0.2024 - learning_rate: 1.0000e-05
Epoch 6/30
164/164 ━━━━━━━━━━━━━━━━━━━━ 138s 842ms/step - accuracy: 0.9508 - loss: 0.1735 - val_accuracy: 0.8750 - val_loss: 0.2234 - learning_rate: 3.0000e-06
Epoch 7/30
164/164 ━━━━━━━━━━━━━━━━━━━━ 140s 852ms/step - accuracy: 0.9542 - loss: 0.1734 - val_accuracy: 0.8750 - val_loss: 0.2383 - learning_rate: 3.0000e-06
Epoch 8/30
164/164 ━━━━━━━━━━━━━━━━━━━━ 140s 851ms/

In [ ]:
# ============================================
# 9. EVALUATE MODEL
# ============================================

loss, accuracy = model.evaluate(test_data)

print("\nTest Accuracy:", accuracy)

20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 367ms/step - accuracy: 0.8766 - loss: 0.4027

Test Accuracy: 0.8766025900840759


In [ ]:
# ============================================
# 10. CONFUSION MATRIX
# ============================================

y_pred_prob = model.predict(test_data)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()
y_true = test_data.classes

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
from sklearn.metrics import classification_report
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


Classification Report:

              precision    recall  f1-score   support

      NORMAL       0.79      0.92      0.85       234
   PNEUMONIA       0.95      0.85      0.90       390

    accuracy                           0.88       624
   macro avg       0.87      0.89      0.87       624
weighted avg       0.89      0.88      0.88       624



In [ ]:
# ============================================
# 11. SAVE FINAL MODEL
# ============================================

model.save("pneumonia_resnet50_best_final.h5")

print("\nFinal model saved successfully.")


Final model saved successfully.
